# Reviewer 2 未対応ギャップの追加実験ノートブック（v5）

`g1_sensory_ablation4_with_baseline8.ipynb` に追加すると実行時間が長くなりすぎるため、
以下の3点だけに絞った、独立した新規ノートブックです。

- **セクションA**: `default_joint_pos`（実際のデフォルト姿勢の値）との比較による、
  「デフォルト姿勢への崩壊」機序の検証を、全8 seedに拡張（Major Comment 8 対応）。
- **セクションB**: heading（初期姿勢の向き）そのものを変える実験（Major Comment 7 対応の一部）。
- **セクションC**: mirrored-state test（Major Comment 7 対応、実験的・要注意）。

**重要**: セクションCは、他のセクションと異なり、このノートブック内で一度も実行・検証されて
いない未確認のAPI（joint状態を直接書き換えるメソッド）と、符号反転についての仮定
（要検証）を含みます。必ずセクションC内の注意書きに従い、まずAPI確認セルとサニティ
チェックセルの結果を確認してから、メイン実験に進んでください。


## 0. セットアップ


In [ ]:
#mount
from google.colab import drive
drive.mount('/content/drive')

!cp -r "/content/drive/MyDrive/g1_cross_seed_logs/logs" /content/ && echo "復元しました" || echo "復元に失敗しました。パスを確認してください"


Mounted at /content/drive
復元しました


In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = ""
os.environ["WANDB_MODE"] = "offline"

# 既存notebookと同じ理由(Colab側GPUドライバのCUDA13.0報告とtorch(cu124)側NVRTCランタイムの
# 不一致によるlibnvrtc-builtins.soエラー)を回避するため、TorchDynamo/Inductorを無効化する。
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"

%pip install -q "mjlab==1.2.0" "mujoco==3.5.0" "warp-lang==1.12.1" "numpy>=2.0,<2.3" scipy mediapy pandas

import torch
assert torch.cuda.is_available(), "GPUランタイムに切り替えてください"
print("install done:", torch.cuda.get_device_name(0))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.4/136.4 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.2/84.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.5/595.5 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.0/750.0 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24

In [ ]:
import math
import glob
import re
import os
import csv
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import asdict

from mjlab.envs import ManagerBasedRlEnv
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg
from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper

TASK = "Mjlab-Velocity-Flat-Unitree-G1"
DEVICE = "cuda:0"
ENV_SEED = 100
RIGHT_FACING_YAW_RAD = -math.pi / 2

JOINT_POS_START = 9
JOINT_VEL_START = 38

# JOINT_NAMES と、全29関節のデフォルト角度(default_joint_pos)を確定させる。
# default_joint_posは、既存notebook(g1_sensory_ablation4_with_baseline.ipynb)の
# 機序解析セクションで実際に使用・実行・検証済みの
# raw_env.scene["robot"].data.default_joint_pos というAPIをそのまま使用する。
_probe_cfg = load_env_cfg(TASK, play=True)
_probe_cfg.scene.num_envs = 1
_probe_cfg.seed = ENV_SEED
_probe_env = ManagerBasedRlEnv(cfg=_probe_cfg, device=DEVICE, render_mode=None)
JOINT_NAMES = list(_probe_env.scene["robot"].joint_names)
assert len(JOINT_NAMES) == 29
default_joint_pos_all_rad = _probe_env.scene["robot"].data.default_joint_pos[0].detach().cpu().numpy().copy()
_probe_env.close()
del _probe_env
torch.cuda.empty_cache()

print("JOINT_NAMES(29関節)とデフォルト角度:")
for i, name in enumerate(JOINT_NAMES):
    print(f"  [{i:2d}] {name:28s} default={math.degrees(default_joint_pos_all_rad[i]):6.1f}deg")

LEFT_HALF_JOINTS = [n for n in JOINT_NAMES if n.startswith("left_")]
RIGHT_HALF_JOINTS = [n for n in JOINT_NAMES if n.startswith("right_")]
TRUNK_JOINTS = ["waist_yaw_joint", "waist_roll_joint", "waist_pitch_joint"]
assert len(LEFT_HALF_JOINTS) == 13 and len(RIGHT_HALF_JOINTS) == 13
assert set(LEFT_HALF_JOINTS) | set(RIGHT_HALF_JOINTS) | set(TRUNK_JOINTS) == set(JOINT_NAMES)

_cond_16r = list(set(RIGHT_HALF_JOINTS) | set(TRUNK_JOINTS))
_cond_16l = list(set(LEFT_HALF_JOINTS) | set(TRUNK_JOINTS))
BOUNDARY16_CONDITIONS = {
    "right_half+trunk": [JOINT_NAMES.index(n) for n in _cond_16r],
    "left_half+trunk": [JOINT_NAMES.index(n) for n in _cond_16l],
}
print("BOUNDARY16_CONDITIONS:", {k: len(v) for k, v in BOUNDARY16_CONDITIONS.items()})


def find_checkpoint(seed):
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    if not run_dirs:
        return None
    return max(Path(run_dirs[-1]).glob("model_*.pt"),
               key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))


def blind_observation(obs, blind_joint_indices, mode="zero", level=0.0):
    """既存notebookのblind_observationと完全に同じ実装。"""
    obs = obs.clone()
    actor_obs = obs["actor"].clone()
    for idx in blind_joint_indices:
        pos_col = JOINT_POS_START + idx
        vel_col = JOINT_VEL_START + idx
        if mode == "zero":
            actor_obs[:, pos_col] = 0.0
            actor_obs[:, vel_col] = 0.0
        elif mode == "scale":
            actor_obs[:, pos_col] = actor_obs[:, pos_col] * level
            actor_obs[:, vel_col] = actor_obs[:, vel_col] * level
        else:
            raise ValueError(f"未知のmode: {mode}")
    obs["actor"] = actor_obs
    return obs


Setting seed: 100
Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.context.runtime` will soon be removed from the public API. It can still be accessed from `warp._src.context.runtime` but might be changed or removed without notice.
Module mujoco_warp._src.smooth f756680 load on device 'cuda:0' took 7211.80 ms  (compiled)
Module mujoco_warp._src.collision_driver 4d8b905 load on device 'cuda:0' took 146.96 ms  (compiled)
Module _nxn_broadphase__locals__kernel_e916423b e916423 load on device 'cuda:0' took 522.69 ms  (compiled)
Module _primitive_narrowphase__locals__primitive_narrowphase_f33437d9 f33437d load on device 'cuda:0' took 1180.01 ms  (compiled)
Module mujoco_warp._src.constraint 63c97cb load on device 'cuda:0' took 3902.63 ms  (compiled)
Module mujoco_warp._src.forward 7114b3d load 

## A. デフォルト姿勢との比較による「崩壊機序」の検証・全8seedへの拡張(Major Comment 8)

原論文の機序解析(セクション3.6)は、4 seedのうち3件の転倒例のみが対象でした
(Limitations §5項目2で著者自身も明記している限界)。ここでは同じ検証
(`default_joint_pos`との比較)を、全8 seedに拡張して実施します。あわせて、
reviewerが要求する「action・PD目標・トルク・接地・姿勢を含む連続した転倒前軌道」も
同時に記録します。


In [ ]:
MECH_DIR = "mechanism_logs_full8seed"
os.makedirs(MECH_DIR, exist_ok=True)
MECH_SEEDS = list(range(1, 9))
MECH_SPEED = 0.5


def run_mechanism_rollout(checkpoint_path, blind_joint_indices, forward_speed_mps,
                           env_seed=ENV_SEED, steps=250):
    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.seed = env_seed
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)

    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(checkpoint_path), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)

    robot = raw_env.scene["robot"]
    contact_sensor = raw_env.scene["feet_ground_contact"]

    forward_cmd = torch.tensor([[forward_speed_mps, 0.0, 0.0]], device=DEVICE)
    obs = env.get_observations()

    history = []
    fell_over = False
    fell_step = None
    for step in range(steps):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd
        if blind_joint_indices:
            obs = blind_observation(obs, blind_joint_indices)
        with torch.no_grad():
            action = policy(obs)
        obs, _, dones, _ = env.step(action)

        record = {
            "t": step,
            "action": action[0].detach().cpu().numpy().copy(),
            "joint_pos": robot.data.joint_pos[0].detach().cpu().numpy().copy(),
            "joint_vel": robot.data.joint_vel[0].detach().cpu().numpy().copy(),
            "joint_pos_target": robot.data.joint_pos_target[0].detach().cpu().numpy().copy(),
            "joint_effort_target": robot.data.joint_effort_target[0].detach().cpu().numpy().copy(),
            "root_lin_vel_w": robot.data.root_link_lin_vel_w[0].detach().cpu().numpy().copy(),
            "foot_contact_found": contact_sensor.data.found[0].detach().cpu().numpy().copy(),
            "foot_contact_force": contact_sensor.data.force[0].detach().cpu().numpy().copy(),
        }
        history.append(record)

        fell = bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0])
        if fell:
            fell_over = True
            fell_step = step
            break

    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()
    return {"fell": fell_over, "fell_step": fell_step, "history": history}


for cond_name, joint_idx in BOUNDARY16_CONDITIONS.items():
    safe_name = cond_name.replace("+", "_")
    for seed in MECH_SEEDS:
        out_path = os.path.join(MECH_DIR, f"{safe_name}_seed{seed}.pkl")
        if os.path.exists(out_path):
            print(f"{cond_name} / seed{seed}: 既存ファイルをスキップ")
            continue
        ckpt = find_checkpoint(seed)
        if ckpt is None:
            print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
            continue
        result = run_mechanism_rollout(ckpt, joint_idx, MECH_SPEED)
        with open(out_path, "wb") as f:
            pickle.dump(result, f)
        print(f"{cond_name} / seed{seed}: fell={result['fell']}, fell_step={result['fell_step']}")

print("done:", MECH_DIR)


Setting seed: 100

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 1      |
| Environment device     | cuda:0 |
| Environment seed       | 100    |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 2 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+--------+----------------------------+
| Index  | Name                       |
+--------+----------------------------+
|   0    | reset_base                 |
|   1    | reset_robot_joints         |
|   2    | randomize_terrain          |
+--------+----------------------------+
+---------------------------------------+
| Active Event Terms in Mode: 'startup' |
+-----------+---------------------------+
|   Index   | Name                      |
+--------

In [ ]:
# --- 解析: 転倒した全試行について、デフォルト角度との差、および
#     直前ステップのpolicy自身の目標角度との差(tracking error)の、両方を報告する ---

default_joint_pos_deg = np.degrees(default_joint_pos_all_rad)

mech_results = []
for cond_name, joint_idx in BOUNDARY16_CONDITIONS.items():
    safe_name = cond_name.replace("+", "_")
    blinded_set = set(JOINT_NAMES[i] for i in joint_idx)
    for seed in MECH_SEEDS:
        path = os.path.join(MECH_DIR, f"{safe_name}_seed{seed}.pkl")
        if not os.path.exists(path):
            continue
        with open(path, "rb") as f:
            result = pickle.load(f)
        if not result["fell"]:
            print(f"{cond_name} / seed{seed}: 転倒せず、対象外")
            continue

        last = result["history"][-1]
        jp_deg = np.degrees(last["joint_pos"])
        jpt_deg = np.degrees(last["joint_pos_target"])

        diff_from_default = jp_deg - default_joint_pos_deg
        diff_from_target = jp_deg - jpt_deg

        for i, name in enumerate(JOINT_NAMES):
            mech_results.append({
                "condition": cond_name, "seed": seed, "joint": name,
                "was_blinded": name in blinded_set,
                "abs_diff_from_default_deg": abs(diff_from_default[i]),
                "abs_diff_from_target_deg": abs(diff_from_target[i]),
            })

mech_df = pd.DataFrame(mech_results)
if len(mech_df) == 0:
    print("転倒事例が1件もありませんでした(全8seedとも0.5m/sで転倒しなかった可能性があります)。")
else:
    summary = mech_df.groupby(["condition", "seed", "was_blinded"])[
        ["abs_diff_from_default_deg", "abs_diff_from_target_deg"]
    ].mean().round(2)
    print("=== 転倒直前の終端関節角度: デフォルト角度との差 / policy自身の目標角度との差(度) ===")
    print(summary)
    print()
    print("was_blinded=False(感覚が健在だった関節)のabs_diff_from_default_degが")
    print("0に近ければ近いほど、「感覚喪失側だけでなく健在側の関節までデフォルトに")
    print("収束する」という原論文の主張が、全8seedでも再現されたことになる。")
    mech_df.to_csv("g1_mechanism_full8seed_summary.csv", index=False)


right_half+trunk / seed2: 転倒せず、対象外
right_half+trunk / seed3: 転倒せず、対象外
right_half+trunk / seed7: 転倒せず、対象外
left_half+trunk / seed1: 転倒せず、対象外
left_half+trunk / seed2: 転倒せず、対象外
left_half+trunk / seed3: 転倒せず、対象外
left_half+trunk / seed5: 転倒せず、対象外
left_half+trunk / seed6: 転倒せず、対象外
left_half+trunk / seed7: 転倒せず、対象外
left_half+trunk / seed8: 転倒せず、対象外
=== 転倒直前の終端関節角度: デフォルト角度との差 / policy自身の目標角度との差(度) ===
                                   abs_diff_from_default_deg  \
condition        seed was_blinded                              
left_half+trunk  4    False                              0.0   
                      True                               0.0   
right_half+trunk 1    False                              0.0   
                      True                               0.0   
                 4    False                              0.0   
                      True                               0.0   
                 5    False                              0.0   
                      True 

## B. Heading(初期姿勢の向き)そのものを変える実験(Major Comment 7 対応の一部)

これまでの環境seedの変化(companion実験)は、初期状態の"サンプリングされた実現値"を
変えるものでした。reviewerが要求しているのは、それとは独立に、heading(初期姿勢の向き
そのもの)を体系的に変えることです。`pose_range["yaw"]`を、元の値
(`RIGHT_FACING_YAW_RAD = -90°`)以外の複数の値に設定して比較します。


In [ ]:
HEADING_CSV = "g1_heading_sweep.csv"
HEADING_VALUES_RAD = {
    "original(-90deg)": RIGHT_FACING_YAW_RAD,
    "0deg": 0.0,
    "+90deg": math.pi / 2,
    "180deg": math.pi,
    "-45deg": -math.pi / 4,
}
HEADING_SEEDS = list(range(1, 9))
HEADING_SPEED = 0.5


def run_heading_rollout(checkpoint_path, blind_joint_indices, forward_speed_mps, yaw_rad,
                         env_seed=ENV_SEED, steps=250):
    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.seed = env_seed
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (yaw_rad, yaw_rad)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)

    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(checkpoint_path), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)

    forward_cmd = torch.tensor([[forward_speed_mps, 0.0, 0.0]], device=DEVICE)
    obs = env.get_observations()
    fell_over = False
    for step in range(steps):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd
        if blind_joint_indices:
            obs = blind_observation(obs, blind_joint_indices)
        with torch.no_grad():
            action = policy(obs)
        obs, _, dones, _ = env.step(action)
        fell = bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0])
        if fell:
            fell_over = True
            break

    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()
    return fell_over


def _load_done_keys_heading(path):
    done = set()
    if os.path.exists(path):
        with open(path, "r", newline="") as f:
            for row in csv.DictReader(f):
                done.add((row["condition"], row["heading_label"], int(row["seed"])))
    return done


file_exists = os.path.exists(HEADING_CSV)
done_keys = _load_done_keys_heading(HEADING_CSV)

with open(HEADING_CSV, "a", newline="") as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(["condition", "heading_label", "seed", "speed_mps", "fell"])
    for cond_name, joint_idx in BOUNDARY16_CONDITIONS.items():
        for heading_label, yaw_rad in HEADING_VALUES_RAD.items():
            for seed in HEADING_SEEDS:
                key = (cond_name, heading_label, seed)
                if key in done_keys:
                    continue
                ckpt = find_checkpoint(seed)
                if ckpt is None:
                    continue
                fell = run_heading_rollout(ckpt, joint_idx, HEADING_SPEED, yaw_rad)
                writer.writerow([cond_name, heading_label, seed, HEADING_SPEED, int(fell)])
                f.flush()
                done_keys.add(key)
                print(f"{cond_name} / {heading_label} / seed{seed}: fell={fell}")

print("done:", HEADING_CSV)


ストリーミング出力は最後の 5000 行に切り捨てられました。
| Index | Name      | Time Out |
+-------+-----------+----------+
|   0   | time_out  |   True   |
|   1   | fell_over |  False   |
+-------+-----------+----------+

[INFO] <RewardManager> contains 14 active terms.
+-----------------------------------------+
|           Active Reward Terms           |
+-------+------------------------+--------+
| Index | Name                   | Weight |
+-------+------------------------+--------+
|   0   | track_linear_velocity  |    2.0 |
|   1   | track_angular_velocity |    2.0 |
|   2   | upright                |    1.0 |
|   3   | pose                   |    1.0 |
|   4   | body_ang_vel           |  -0.05 |
|   5   | angular_momentum       |  -0.02 |
|   6   | dof_pos_limits         |   -1.0 |
|   7   | action_rate_l2         |   -0.1 |
|   8   | air_time               |    0.0 |
|   9   | foot_clearance         |   -2.0 |
|   10  | foot_swing_height      |  -0.25 |
|   11  | foot_slip              |   -0.1 |
|   1

In [ ]:
heading_df = pd.read_csv(HEADING_CSV)
pivot = heading_df.pivot_table(index="heading_label", columns="condition", values="fell", aggfunc="mean") * 100
print("=== heading(初期姿勢の向き)ごとの転倒率(%) ===")
print(pivot.round(1))
print()
print("headingを変えても両条件の転倒率の順序(right_half+trunkの方が高い)が保たれるか、")
print("それとも特定のheadingで逆転・大きく変動するかを確認する。")


=== heading(初期姿勢の向き)ごとの転倒率(%) ===
condition         left_half+trunk  right_half+trunk
heading_label                                      
+90deg                       12.5              62.5
-45deg                       12.5              62.5
0deg                         12.5              62.5
180deg                       12.5              62.5
original(-90deg)             12.5              62.5

headingを変えても両条件の転倒率の順序(right_half+trunkの方が高い)が保たれるか、
それとも特定のheadingで逆転・大きく変動するかを確認する。


## C. Mirrored-state test(Major Comment 7 対応、実験的・要注意)

**重要な注意**: このセクションのコードは、他のセクションと異なり、このノートブック内で
一度も実行・検証されていない未確認のAPI(joint状態を直接書き換えるメソッド)を使用します。
また、左右の関節をミラーする際の符号反転規則(pitch関節は符号を保持、roll/yaw関節は
符号を反転するという、一般的な人型ロボットの関節軸定義の慣習を仮定したもの)も、
**この環境で実際に検証されていない仮定**です。

必ず、(1)まずAPI確認セルの出力を確認し、必要なら`JOINT_STATE_WRITE_METHOD`の値を
実際のメソッド名に修正してから、(2)無傷条件でのサニティチェックセルを実行し、
`mirrored=False`と`mirrored=True`の転倒率が大きく異ならないことを確認してから、
(3)メインの実験セルに進んでください。サニティチェックで大きな差が出た場合、
符号反転規則を見直す必要があります。


In [ ]:
# --- API確認: joint状態を直接書き換えるメソッドが、実際にどう呼ばれるかを確認する ---

_probe_cfg_c = load_env_cfg(TASK, play=True)
_probe_cfg_c.scene.num_envs = 1
_probe_env_c = ManagerBasedRlEnv(cfg=_probe_cfg_c, device=DEVICE, render_mode=None)
_robot_c = _probe_env_c.scene["robot"]

print("=== robotオブジェクトの、joint状態の書き込みに関連しそうなメソッド ===")
print([a for a in dir(_robot_c) if "write" in a.lower() or "joint" in a.lower()])
print()
print("=== robot.dataの、joint状態に関連する属性 ===")
print([a for a in dir(_robot_c.data) if "joint" in a.lower()])

_probe_env_c.close()
del _probe_env_c
torch.cuda.empty_cache()

print()
print("上の出力を見て、次のセルのJOINT_STATE_WRITE_METHODが正しいメソッド名になっているか")
print("確認してください(IsaacLab系のAPI慣習に基づく write_joint_state_to_sim という")
print("名前を仮に置いていますが、このAPIはこの環境で未検証です)。")



+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 1      |
| Environment device     | cuda:0 |
| Environment seed       | None   |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 2 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+--------+----------------------------+
| Index  | Name                       |
+--------+----------------------------+
|   0    | reset_base                 |
|   1    | reset_robot_joints         |
|   2    | randomize_terrain          |
+--------+----------------------------+
+---------------------------------------+
| Active Event Terms in Mode: 'startup' |
+-----------+---------------------------+
|   Index   | Name                      |
+-----------+--------------

In [ ]:
# --- 関節ペアの対応表と、符号反転ルールを構築する ---
# 【仮定・要検証】pitch関節(矢状面の動き)は符号を反転せず、roll・yaw関節
# (前額面・水平面の動き)は左右ミラーで符号を反転する、という一般的な人型ロボットの
# 関節軸定義の慣習を仮定している。この仮定が実際のG1の関節軸定義と一致しているかは、
# 次のセルのサニティチェック(無傷条件)で確認する。

def mirror_pair_and_sign(name):
    if name.startswith("left_"):
        mirror_name = name.replace("left_", "right_", 1)
    elif name.startswith("right_"):
        mirror_name = name.replace("right_", "left_", 1)
    else:
        mirror_name = name  # 体幹(waist)関節は自分自身とペア
    sign = -1.0 if ("_roll" in name or "_yaw" in name) else 1.0
    return mirror_name, sign


MIRROR_INDEX = np.zeros(29, dtype=int)
MIRROR_SIGN = np.zeros(29, dtype=float)
for i, name in enumerate(JOINT_NAMES):
    mirror_name, sign = mirror_pair_and_sign(name)
    MIRROR_INDEX[i] = JOINT_NAMES.index(mirror_name)
    MIRROR_SIGN[i] = sign

print("関節ミラー対応表:")
for i, name in enumerate(JOINT_NAMES):
    print(f"  {name:28s} -> {JOINT_NAMES[MIRROR_INDEX[i]]:28s} (sign={MIRROR_SIGN[i]:+.0f})")


def mirror_joint_state(joint_pos, joint_vel):
    """joint_pos, joint_vel: (29,) のnumpy配列(1環境分)。ミラーした配列を返す。
    ベース(体幹全体)の向き・姿勢はミラーしない(関節角度のみの、部分的なミラー)。"""
    mirrored_pos = joint_pos[MIRROR_INDEX] * MIRROR_SIGN
    mirrored_vel = joint_vel[MIRROR_INDEX] * MIRROR_SIGN
    return mirrored_pos, mirrored_vel


JOINT_STATE_WRITE_METHOD = "write_joint_state_to_sim"  # 前のセルの出力を見て、必要なら変更してください

# 【修正】reset直後は、joint_pos が default_joint_pos に厳密に一致しており
# (診断セルで確認済み: 差はぴったり0)、かつ default_joint_pos 自体が左右完全対称
# (診断セルで確認済み: 例 left_shoulder_roll=+11.5deg, right_shoulder_roll=-11.5deg)
# であるため、reset直後にミラー処理をしても mirror_joint_state(default) == default と
# なり、数学的に無意味な操作(no-op)になってしまう。
# そこで、まず MIRROR_WARMUP_STEPS ステップだけ通常通り前進歩行させ、歩容が
# 左右非対称になった(遊脚期・片脚支持期などの)状態を作ってから、その時点で
# ミラー処理を行う。ウォームアップ中はmirrored_initial_stateの値に関わらず
# 完全に同一の軌道をたどるため(同じenv_seed・同じpolicy・分岐はまだ発生していない)、
# mirrored=Falseとmirrored=Trueの比較は「ウォームアップ後の同一状態」を起点とした
# 公平な比較になる。
MIRROR_WARMUP_STEPS = 25  # 0.02s/step x 25 = 0.5秒。歩容周期の途中(非対称な姿勢)を狙う


def run_mirrored_rollout(checkpoint_path, blind_joint_indices, forward_speed_mps, mirror_initial_state,
                          env_seed=ENV_SEED, steps=250, warmup_steps=MIRROR_WARMUP_STEPS):
    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.seed = env_seed
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)

    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(checkpoint_path), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)

    robot = raw_env.scene["robot"]
    obs = env.get_observations()

    forward_cmd = torch.tensor([[forward_speed_mps, 0.0, 0.0]], device=DEVICE)

    # --- ウォームアップ: 左右非対称な歩容状態を作る(mirrored変数に関わらず共通) ---
    fell_during_warmup = False
    for _ in range(warmup_steps):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd
        with torch.no_grad():
            action = policy(obs)
        obs, _, dones, _ = env.step(action)
        fell = bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0])
        if fell:
            fell_during_warmup = True
            break

    if fell_during_warmup:
        # ウォームアップ中に転倒した場合、ミラー処理の適用対象となる状態が存在しない。
        # mirrored=Trueとmirrored=Falseの比較が成立しないため、専用のフラグで区別して返す。
        env.close()
        del runner, policy, env, raw_env
        torch.cuda.empty_cache()
        return True, True  # (fell_over, fell_during_warmup)

    if mirror_initial_state:
        write_fn = getattr(robot, JOINT_STATE_WRITE_METHOD, None)
        if write_fn is None:
            raise AttributeError(
                f"robotに'{JOINT_STATE_WRITE_METHOD}'メソッドが見つかりません。"
                f"前のセクションのAPI確認セルの出力を見て、JOINT_STATE_WRITE_METHODを"
                f"正しいメソッド名に修正してください。"
            )
        current_pos = robot.data.joint_pos[0].detach().cpu().numpy().copy()
        current_vel = robot.data.joint_vel[0].detach().cpu().numpy().copy()
        mirrored_pos, mirrored_vel = mirror_joint_state(current_pos, current_vel)
        mirrored_pos_t = torch.tensor(mirrored_pos, device=DEVICE, dtype=torch.float32).unsqueeze(0)
        mirrored_vel_t = torch.tensor(mirrored_vel, device=DEVICE, dtype=torch.float32).unsqueeze(0)
        env_ids = torch.arange(1, device=DEVICE)
        write_fn(mirrored_pos_t, mirrored_vel_t, env_ids=env_ids)
        # 【重要】env.get_observations()やobservation_manager.compute()はキャッシュを
        # 返すだけで、直接書き込みには反応しないことが診断で確認された。
        # observation_manager.compute_group("actor")を直接呼べば、ライブの物理状態を
        # 反映した観測が得られる(診断セルで検証済み: 最大絶対誤差0.0)。
        fresh_actor_obs = raw_env.observation_manager.compute_group("actor")
        obs = obs.clone()
        obs["actor"] = fresh_actor_obs

    fell_over = False
    for step in range(steps):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd
        if blind_joint_indices:
            obs = blind_observation(obs, blind_joint_indices)
        with torch.no_grad():
            action = policy(obs)
        obs, _, dones, _ = env.step(action)
        fell = bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0])
        if fell:
            fell_over = True
            break

    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()
    return fell_over, False  # (fell_over, fell_during_warmup)


関節ミラー対応表:
  left_hip_pitch_joint         -> right_hip_pitch_joint        (sign=+1)
  left_hip_roll_joint          -> right_hip_roll_joint         (sign=-1)
  left_hip_yaw_joint           -> right_hip_yaw_joint          (sign=-1)
  left_knee_joint              -> right_knee_joint             (sign=+1)
  left_ankle_pitch_joint       -> right_ankle_pitch_joint      (sign=+1)
  left_ankle_roll_joint        -> right_ankle_roll_joint       (sign=-1)
  right_hip_pitch_joint        -> left_hip_pitch_joint         (sign=+1)
  right_hip_roll_joint         -> left_hip_roll_joint          (sign=-1)
  right_hip_yaw_joint          -> left_hip_yaw_joint           (sign=-1)
  right_knee_joint             -> left_knee_joint              (sign=+1)
  right_ankle_pitch_joint      -> left_ankle_pitch_joint       (sign=+1)
  right_ankle_roll_joint       -> left_ankle_roll_joint        (sign=-1)
  waist_yaw_joint              -> waist_yaw_joint              (sign=-1)
  waist_roll_joint             -> waist_r

In [ ]:
# --- 診断: write_joint_state_to_sim が実際に反映されているかを確認する ---
# 実行結果(全seedでmirrored=Falseとmirrored=Trueの転倒/非転倒が1件も違わなかった)を見ると、
# この書き込みが実際には反映されていない疑いが強い。まずこの点を直接確認する。

_diag_cfg = load_env_cfg(TASK, play=True)
_diag_cfg.scene.num_envs = 1
_diag_cfg.seed = ENV_SEED
_diag_agent_cfg = load_rl_cfg(TASK)
_diag_raw_env = ManagerBasedRlEnv(cfg=_diag_cfg, device=DEVICE, render_mode=None)
_diag_env = RslRlVecEnvWrapper(_diag_raw_env, clip_actions=_diag_agent_cfg.clip_actions)
_diag_robot = _diag_raw_env.scene["robot"]

_ = _diag_env.get_observations()
pos_before = _diag_robot.data.joint_pos[0].detach().cpu().numpy().copy()
print("書き込み前のjoint_pos(先頭5関節、度):", np.degrees(pos_before[:5]))

# 明らかに現在の状態と異なる、テスト用のダミー値(全関節+30度)を書き込んでみる
test_pos = torch.tensor(pos_before + math.radians(30), device=DEVICE, dtype=torch.float32).unsqueeze(0)
test_vel = torch.zeros_like(test_pos)
env_ids = torch.arange(1, device=DEVICE)

write_fn = getattr(_diag_robot, "write_joint_state_to_sim")
write_fn(test_pos, test_vel, env_ids=env_ids)

pos_after_write_raw = _diag_robot.data.joint_pos[0].detach().cpu().numpy().copy()
print("write直後のrobot.data.joint_pos(先頭5関節、度):", np.degrees(pos_after_write_raw[:5]))

obs_after = _diag_env.get_observations()
pos_after_obs = _diag_robot.data.joint_pos[0].detach().cpu().numpy().copy()
print("get_observations()後のrobot.data.joint_pos(先頭5関節、度):", np.degrees(pos_after_obs[:5]))

actor_obs_slice = obs_after["actor"][0, JOINT_POS_START:JOINT_POS_START + 5].detach().cpu().numpy()
print("get_observations()後のobs['actor'](joint_pos部分、生値):", actor_obs_slice)

_diag_env.close()
del _diag_env, _diag_raw_env
torch.cuda.empty_cache()

print()
print("判定基準:")
print("(1)『write直後』の値が『書き込み前』+30度に近ければ、write_joint_state_to_sim自体は")
print("   simの状態に反映されている。")
print("(2)『get_observations()後』の値が『書き込み前』の値に戻ってしまっている場合、")
print("   get_observations()の内部で別ソースから状態が再取得されており、")
print("   ミラー実験の書き込みが無効化されていることになる。")
print("(3) obs['actor']のjoint_pos部分の生値が(1)(2)のjoint_posと整合しているかも確認する")
print("   (policyに実際に渡る値がここで確定するため)。")


Setting seed: 100

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 1      |
| Environment device     | cuda:0 |
| Environment seed       | 100    |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 2 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+--------+----------------------------+
| Index  | Name                       |
+--------+----------------------------+
|   0    | reset_base                 |
|   1    | reset_robot_joints         |
|   2    | randomize_terrain          |
+--------+----------------------------+
+---------------------------------------+
| Active Event Terms in Mode: 'startup' |
+-----------+---------------------------+
|   Index   | Name                      |
+--------

In [ ]:
# --- 追加診断: get_observations()がキャッシュを返しているだけなのか、
#     observation_managerを直接呼べばsimの現在状態を反映するのかを確認する ---
# 前のセルの結果(write後もobs['actor']のjoint_pos部分がゼロのまま変化しない)から、
# get_observations()がsimをライブに再計算せず、reset時などに作られたキャッシュを
# 返している疑いが強い。ここでは、ラップ前のraw_envが持つobservation_managerを
# 直接呼び出し(compute()系のメソッドを名前を変えて何通りか試す)、書き込み後の
# 状態が反映されるかどうかを確認する。

_diag2_cfg = load_env_cfg(TASK, play=True)
_diag2_cfg.scene.num_envs = 1
_diag2_cfg.seed = ENV_SEED
_diag2_agent_cfg = load_rl_cfg(TASK)
_diag2_raw_env = ManagerBasedRlEnv(cfg=_diag2_cfg, device=DEVICE, render_mode=None)
_diag2_env = RslRlVecEnvWrapper(_diag2_raw_env, clip_actions=_diag2_agent_cfg.clip_actions)
_diag2_robot = _diag2_raw_env.scene["robot"]

_ = _diag2_env.get_observations()
pos_before2 = _diag2_robot.data.joint_pos[0].detach().cpu().numpy().copy()
print("書き込み前のjoint_pos(先頭5関節、度):", np.degrees(pos_before2[:5]))

test_pos2 = torch.tensor(pos_before2 + math.radians(30), device=DEVICE, dtype=torch.float32).unsqueeze(0)
test_vel2 = torch.zeros_like(test_pos2)
env_ids2 = torch.arange(1, device=DEVICE)
_diag2_robot.write_joint_state_to_sim(test_pos2, test_vel2, env_ids=env_ids2)
print("write直後のjoint_pos(先頭5関節、度):",
      np.degrees(_diag2_robot.data.joint_pos[0].detach().cpu().numpy()[:5]))

# --- observation_managerを見つけて、直接呼べるメソッドを列挙する ---
obs_mgr = getattr(_diag2_raw_env, "observation_manager", None)
print()
print("=== raw_env.observation_manager の有無とメソッド一覧 ===")
print("observation_manager:", obs_mgr)
if obs_mgr is not None:
    candidate_methods = [name for name in dir(obs_mgr)
                          if not name.startswith("_") and callable(getattr(obs_mgr, name))]
    print("呼び出し可能な公開メソッド:", candidate_methods)

# --- 候補メソッドを順に試し、書き込み後の状態を反映した再計算ができるか確認する ---
COMPUTE_METHOD_CANDIDATES = ["compute", "compute_group", "compute_observations", "get_observations"]

for method_name in COMPUTE_METHOD_CANDIDATES:
    if obs_mgr is None:
        break
    fn = getattr(obs_mgr, method_name, None)
    if fn is None:
        print(f"[{method_name}] メソッドが存在しません。スキップ。")
        continue
    try:
        if method_name == "compute_group":
            result = fn("actor")
        else:
            result = fn()
    except Exception as e:
        print(f"[{method_name}] 呼び出しでエラー: {type(e).__name__}: {e}")
        continue

    # 戻り値の形(dict / tensor)に応じてactorのjoint_pos部分を取り出す
    try:
        if isinstance(result, dict):
            actor_tensor = result.get("actor", None)
        else:
            actor_tensor = result
        if actor_tensor is None:
            print(f"[{method_name}] 戻り値から'actor'を取り出せませんでした。型: {type(result)}")
            continue
        slice_vals = actor_tensor[0, JOINT_POS_START:JOINT_POS_START + 5].detach().cpu().numpy()
        print(f"[{method_name}] joint_pos部分(先頭5関節、生値): {slice_vals}")
    except Exception as e:
        print(f"[{method_name}] 戻り値の解析でエラー: {type(e).__name__}: {e}")

# --- 参考: raw_envそのものに再計算メソッドがないかも確認する ---
print()
print("=== raw_env自体の候補メソッド ===")
env_candidate_methods = [name for name in dir(_diag2_raw_env)
                          if ("obs" in name.lower()) and not name.startswith("_")]
print(env_candidate_methods)

_diag2_env.close()
del _diag2_env, _diag2_raw_env
torch.cuda.empty_cache()

print()
print("判定基準:")
print("- 上記のいずれかのメソッドで、joint_pos部分の生値が『書き込み前とは明確に異なる値』")
print("  (かつ非ゼロ)になっていれば、そのメソッドを使って手動でobsを再計算すればよい")
print("  ことになる(mirror実験の修正方法が確定する)。")
print("- どのメソッドを試してもゼロ、あるいは書き込み前の値のままなら、")
print("  この環境で『sim状態を直接書き換えてpolicy入力に反映させる』こと自体が")
print("  困難である可能性が高く、mirror実験は別の実装方針(例えばreset時のyaw")
print("  パラメータ経由など、既にセクションBで使っている正攻法)に切り替える")
print("  必要がある。")


Setting seed: 100

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 1      |
| Environment device     | cuda:0 |
| Environment seed       | 100    |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 2 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+--------+----------------------------+
| Index  | Name                       |
+--------+----------------------------+
|   0    | reset_base                 |
|   1    | reset_robot_joints         |
|   2    | randomize_terrain          |
+--------+----------------------------+
+---------------------------------------+
| Active Event Terms in Mode: 'startup' |
+-----------+---------------------------+
|   Index   | Name                      |
+--------

In [ ]:
# --- 修正の検証: compute_group("actor")を使えば、書き込みが正しく反映されるかを確認する ---
# 前のセルの結果から、joint_pos観測項は「デフォルト姿勢からの相対角度」であり、
# observation_manager.compute_group("actor")を直接呼べばライブの物理状態を反映する
# 可能性が高いことが分かった。ここで、以下の一致を確認して仮説を検証する:
#   (新しいjoint_pos) - (default_joint_pos) が、compute_group("actor")が返す
#   joint_pos部分の値と一致するかどうか。

_diag3_cfg = load_env_cfg(TASK, play=True)
_diag3_cfg.scene.num_envs = 1
_diag3_cfg.seed = ENV_SEED
_diag3_agent_cfg = load_rl_cfg(TASK)
_diag3_raw_env = ManagerBasedRlEnv(cfg=_diag3_cfg, device=DEVICE, render_mode=None)
_diag3_env = RslRlVecEnvWrapper(_diag3_raw_env, clip_actions=_diag3_agent_cfg.clip_actions)
_diag3_robot = _diag3_raw_env.scene["robot"]

obs0 = _diag3_env.get_observations()
pos_before3 = _diag3_robot.data.joint_pos[0].detach().cpu().numpy().copy()

# 書き込み前ですでにdefault姿勢に近いはずだが、念のため実際のdefault_joint_posとの
# 差を確認しておく(先頭5関節)。
default5 = default_joint_pos_all_rad[:5]
print("書き込み前のjoint_pos - default_joint_pos(先頭5関節、度):",
      np.degrees(pos_before3[:5] - default5))

# 全関節にランダムなオフセット(関節ごとに異なる値)を書き込み、単なる一律+30度の
# 特殊ケースではないことも確認する。
rng = np.random.default_rng(0)
offset_deg = rng.uniform(-20, 20, size=pos_before3.shape)
test_pos3 = torch.tensor(pos_before3 + np.radians(offset_deg), device=DEVICE, dtype=torch.float32).unsqueeze(0)
test_vel3 = torch.zeros_like(test_pos3)
env_ids3 = torch.arange(1, device=DEVICE)
_diag3_robot.write_joint_state_to_sim(test_pos3, test_vel3, env_ids=env_ids3)

new_pos3 = _diag3_robot.data.joint_pos[0].detach().cpu().numpy().copy()
expected_rel = new_pos3 - default_joint_pos_all_rad

fresh_obs = _diag3_raw_env.observation_manager.compute_group("actor")
actual_rel = fresh_obs[0, JOINT_POS_START:JOINT_POS_START + 29].detach().cpu().numpy()

diff = np.abs(expected_rel - actual_rel)
print()
print("期待値(新joint_pos - default_joint_pos)と、compute_group(\"actor\")の")
print("joint_pos部分(29関節全て)との最大絶対誤差:", diff.max())
print("先頭5関節の期待値(度):", np.degrees(expected_rel[:5]))
print("先頭5関節の実測値(度):", np.degrees(actual_rel[:5]))

_diag3_env.close()
del _diag3_env, _diag3_raw_env
torch.cuda.empty_cache()

print()
print("判定基準: 最大絶対誤差が十分小さければ(例えば1e-3 rad未満)、")
print("compute_group(\"actor\")を使うことで、write_joint_state_to_sim による書き込みを")
print("正しく観測に反映できることが確認できる。これにより、run_mirrored_rollout()内の")
print("get_observations()をcompute_group(\"actor\")に置き換える修正の妥当性が裏付けられる。")


Setting seed: 100

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 1      |
| Environment device     | cuda:0 |
| Environment seed       | 100    |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 2 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+--------+----------------------------+
| Index  | Name                       |
+--------+----------------------------+
|   0    | reset_base                 |
|   1    | reset_robot_joints         |
|   2    | randomize_terrain          |
+--------+----------------------------+
+---------------------------------------+
| Active Event Terms in Mode: 'startup' |
+-----------+---------------------------+
|   Index   | Name                      |
+--------

In [ ]:
# --- サニティチェック: 無傷(ablationなし)条件で、ミラー処理そのものが
#     虚偽の非対称性を生み出していないかを確認する ---
# mirror_joint_state()の符号仮定が誤っていた場合、無傷条件でも通常の初期状態と
# ミラーした初期状態とで、転倒率や挙動にsystematicな差が生じるはずである。
# 差が見られなければ、少なくとも「大きく壊れた」ミラー実装ではないと確認できる
# (これは十分条件ではなく必要条件のチェックである点に注意)。
#
# 【v2】ウォームアップ後にミラー処理を行う修正版に合わせ、CSVファイル名を
# 新しくした(古い、reset直後にミラーしていた実行結果と混在させないため)。
# fell_during_warmupがTrueの試行(ミラー処理を適用する前に転倒した試行)は、
# mirrored=False/Trueの比較が成立しないため、集計から除外する。

SANITY_CSV = "g1_mirror_sanity_check_baseline_v2.csv"
SANITY_SEEDS = list(range(1, 9))
SANITY_SPEED = 0.5

file_exists = os.path.exists(SANITY_CSV)
with open(SANITY_CSV, "a", newline="") as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(["mirrored", "seed", "speed_mps", "fell", "fell_during_warmup"])
    for mirrored in [False, True]:
        for seed in SANITY_SEEDS:
            ckpt = find_checkpoint(seed)
            if ckpt is None:
                continue
            fell, fell_during_warmup = run_mirrored_rollout(ckpt, [], SANITY_SPEED, mirror_initial_state=mirrored)
            writer.writerow([int(mirrored), seed, SANITY_SPEED, int(fell), int(fell_during_warmup)])
            f.flush()
            print(f"baseline / mirrored={mirrored} / seed{seed}: fell={fell} "
                  f"(fell_during_warmup={fell_during_warmup})")

sanity_df = pd.read_csv(SANITY_CSV)
sanity_df_valid = sanity_df[sanity_df["fell_during_warmup"] == 0]
print()
print("ウォームアップ中に転倒して除外した試行数:", int(sanity_df["fell_during_warmup"].sum()))
print(sanity_df_valid.groupby("mirrored")["fell"].mean())
print()
print("上の2つの転倒率が大きく異なる場合(特にmirrored=Trueの方だけ転倒率が高い場合)、")
print("mirror_joint_state()の符号仮定が誤っている可能性が高い。その場合、")
print("mirror_pair_and_sign()内のroll/yaw/pitchの符号反転ルールを見直してから、")
print("以降のメイン実験(mirrored-state test)を実行すること。")


Setting seed: 100

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 1      |
| Environment device     | cuda:0 |
| Environment seed       | 100    |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 2 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+--------+----------------------------+
| Index  | Name                       |
+--------+----------------------------+
|   0    | reset_base                 |
|   1    | reset_robot_joints         |
|   2    | randomize_terrain          |
+--------+----------------------------+
+---------------------------------------+
| Active Event Terms in Mode: 'startup' |
+-----------+---------------------------+
|   Index   | Name                      |
+--------

In [ ]:
# --- メイン実験: 各16関節条件 x {通常の初期状態, ミラーした初期状態} x 8 seed ---
# サニティチェック(上のセル)で mirrored=False と mirrored=True の転倒率に
# 大きな差がないことを確認してから、このセルを実行すること。
#
# 【v2】ウォームアップ後にミラー処理を行う修正版に合わせ、CSVファイル名を新しくした。

MIRROR_MAIN_CSV = "g1_mirror_state_test_v2.csv"
file_exists = os.path.exists(MIRROR_MAIN_CSV)
with open(MIRROR_MAIN_CSV, "a", newline="") as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(["condition", "mirrored", "seed", "speed_mps", "fell", "fell_during_warmup"])
    for cond_name, joint_idx in BOUNDARY16_CONDITIONS.items():
        for mirrored in [False, True]:
            for seed in SANITY_SEEDS:
                ckpt = find_checkpoint(seed)
                if ckpt is None:
                    continue
                fell, fell_during_warmup = run_mirrored_rollout(
                    ckpt, joint_idx, SANITY_SPEED, mirror_initial_state=mirrored)
                writer.writerow([cond_name, int(mirrored), seed, SANITY_SPEED,
                                  int(fell), int(fell_during_warmup)])
                f.flush()
                print(f"{cond_name} / mirrored={mirrored} / seed{seed}: fell={fell} "
                      f"(fell_during_warmup={fell_during_warmup})")

print("done:", MIRROR_MAIN_CSV)

mirror_result_df = pd.read_csv(MIRROR_MAIN_CSV)
mirror_result_df_valid = mirror_result_df[mirror_result_df["fell_during_warmup"] == 0]
print()
print("ウォームアップ中に転倒して除外した試行数:", int(mirror_result_df["fell_during_warmup"].sum()))
print(mirror_result_df_valid.groupby(["condition", "mirrored"])["fell"].mean() * 100)
print()
print("解釈の指針: mirrored=Falseでの通常の非対称性(right_half+trunkの方が転倒率が高い)")
print("が、mirrored=Trueでも同じ方向のまま(right_half+trunk条件の方が依然として高い)")
print("なら、非対称性は「どちらの解剖学的側を除去したか」に紐づいている可能性が高い。")
print("逆に、mirrored=Trueで非対称性の方向が入れ替わる、あるいは消える場合は、")
print("シミュレーション側の何らかの共有された左右の慣習がアーティファクトとして")
print("寄与している可能性を否定できない。")


ストリーミング出力は最後の 5000 行に切り捨てられました。
|   3   | pose                   |    1.0 |
|   4   | body_ang_vel           |  -0.05 |
|   5   | angular_momentum       |  -0.02 |
|   6   | dof_pos_limits         |   -1.0 |
|   7   | action_rate_l2         |   -0.1 |
|   8   | air_time               |    0.0 |
|   9   | foot_clearance         |   -2.0 |
|   10  | foot_swing_height      |  -0.25 |
|   11  | foot_slip              |   -0.1 |
|   12  | soft_landing           | -1e-05 |
|   13  | self_collisions        |   -1.0 |
+-------+------------------------+--------+

[INFO] <NullCurriculumManager> (inactive)
[INFO] <MetricsManager> contains 1 active terms.
+-------------------------+
|   Active Metrics Terms  |
+-------+-----------------+
| Index | Name            |
+-------+-----------------+
|   0   | mean_action_acc |
+-------+-----------------+

--------------------------------------------------------------------------------
Resolved observation sets: 
	 actor :  ('actor',)
	 critic :  ('critic